# 68 · Độ phân giải cao — **fine-tune backbone ở đúng độ phân giải đó**

## Vì sao

`nb67`: khoảng cách `medium` (**71%** tổng) nằm ở **7 nhãn**, phần lớn **nhãn sạch**
(`prec 0.958–1.000`), và là **cấu trúc mảnh dưới `9.46mm`/patch**:

```
Fissure ~1mm · Pleural Thickening vai mm · Calcification Aorta mm · Granuloma mm
```

Nếu 7 nhãn này lên trung vị medium (`~0.25`): medium `0.2286 → ~0.287` — **vượt vô địch `0.263`**,
macro **`+0.022`**. Đòn bẩy đơn lẻ lớn nhất còn lại. Đội C và D dùng `1024`; ta ở `518`.

## Hai phép đo trước KHÔNG đóng được hướng này

```
51   doi do phan giai o SUY LUAN, dau doc van la ban huan luyen o 518
     -> lech phan phoi (noi suy pos-embed) + pha loang attention     -> -0.0028
66   crop hop bao roi phong to — nhung hop bao phu 79% anh
     -> phong dai THUC chi 1.06x, phep thu mat suc phan giai         -> +0.0003 vo nghia
```

Cả hai đều **không** đổi thứ quan trọng nhất: backbone vẫn là bản fine-tune ở `518`.

## Notebook này đổi độ phân giải ở MỌI TẦNG

```
bac A  fine-tune BACKBONE o SIZE moi (khoi tao tu 38a/a2_E3), LP-FT y het 38a
bac B  dong cung backbone MOI, huan luyen dau doc `tpb` + `mldec`
bac C  so voi 64/r2a tren CUNG thang: tier CHINH THUC + 7 nhan medium
```

Không còn chỗ nào dùng backbone `518`.

## Kích thước phải chia hết cho 14 — và `1022` là shard gốc

RAD-DINO là ViT-B/**14**. `768` và `1024` **không** chia hết; dùng chúng là hỏng lưới patch.

```
 518   37x37 = 1,369 token |  9.46 mm/patch | compute  1.00x   <- moc hien tai
 672   48x48 = 2,304       |  7.29          |          2.03x
 770   55x55 = 3,025       |  6.36          |          3.01x
1022   73x73 = 5,329       |  4.79          |          7.27x   <- MAC DINH, shard GOC
```

`1022` là kích thước **có sẵn của shard** `cxr_1022_letterbox_cxrlt` ⇒ **không resize lần nào**,
không mất chi tiết ở khâu tiền xử lý. Đó là lý do chọn nó thay vì `770`.

## Chi phí (ngoại suy từ `40 ph/epoch` đo được ở `64` @518)

```
                          bac A (backbone)     bac B (dau doc)      tong
SIZE=1022, day du         6 epoch  ~44h        E=10  ~48h/seed      ~92h
SIZE=1022, CHE_DO_THU     3 epoch  ~22h        E=6   ~29h           ~51h   <- MAC DINH
SIZE=770,  CHE_DO_THU     3 epoch   ~9h        E=6   ~12h           ~21h   <- neu muon re hon
```

`CHE_DO_THU = True` chỉ để **trả lời câu hỏi**, so với `64/r2a s42` tại `ep6`. Dương thì mới chạy đủ.

## Cổng khoá trước

```
7 nhan medium: Infiltration · Tortuous Aorta · Pleural Thickening · Adenopathy ·
               Calcification of the Aorta · Fissure · Granuloma
  AP trung binh tang >= +0.03  VA  macro khong tut
-> di tiep: E=10, seed thu 2
```

Khi đọc thì **loại `Infiltration`** khỏi cổng: `prec 0.261`, nhiễu nhãn, độ phân giải không cứu.

## Đĩa và VRAM

```
anh 1022 local: ~49 GB (bang chinh shard, khong nen lai)
bac A: ViT-B full fine-tune @ 5,329 token -> BATCH_BB 16 x ACCUM 15 = EFF 240 (nhu 38a)
bac B: backbone DONG CUNG, 2 anh/mau      -> BATCH 16 x ACCUM 16 = EFF 256 (nhu 64)
OOM thi ha BATCH va nang ACCUM de GIU NGUYEN EFF.
```


## C0 · Config

In [1]:
import os, json, math, time, tarfile, io as _io, re, datetime, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('CXRLT_DATA', '')
    assert DATA_DIR, ('khong phai Colab va chua dat CXRLT_DATA -> gan thang duong dan `planB_out`. '
                      'Nhanh du phong "./planB_out" cu bao SAI nguyen nhan.')
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
SIZE_SHARD  = 1022     # kich thuoc anh trong shard `cxr_1022_letterbox_cxrlt`
SIZE_HI     = 1022     # PHAI chia het cho 14 (ViT-B/14). 1022 = 73x14 = SHARD GOC, khong resize.
CHE_DO_THU  = True     # True: ngan sach rut gon de TRA LOI CAU HOI (A 3 epoch, B E=6, 1 seed)
E_BB        = 3 if CHE_DO_THU else 6      # so epoch fine-tune BACKBONE (bac A)
LR_BB_HEAD, LR_BB_MULT = 5e-4, 0.1        # y het 38a: dau doc 5e-4, backbone x0.1 (LP-FT)
BATCH_BB, ACCUM_BB = 16, 15               # EFF 240, GIU NGUYEN nhu 38a
RUN_TAG = f'r{SIZE_HI}'                   # ket qua ve stage4_nb68/r1022
# giu ten bien `RES57` de khong phai sua hang chuc cho; thu muc thi doi.
RES57 = os.path.join(OUT_DIR, 'stage4_nb68', RUN_TAG); CKPT = os.path.join(RES57, 'ckpt')
P_OOF = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')
# TACH theo notebook: hai tien trinh cung `append` mot file qua Drive FUSE se nuot dong
# cua nhau. 53 va 57 chay song song.
P_TOUCH = os.path.join(OUT_DIR, '_test_touches_nb68.jsonl')
def ensure_res():
    '''Drive FUSE danh roi thu muc RONG — goi TRUOC moi cum ghi (43/38c/38d/41a/41c da dinh).'''
    os.makedirs(RES57, exist_ok=True); os.makedirs(CKPT, exist_ok=True); return RES57
ensure_res()
assert os.path.isdir(OUT_DIR) and os.listdir(OUT_DIR), f'{OUT_DIR} khong doc duoc -> mount hong'
json.dump({'run_tag': RUN_TAG, 'notebook': '68'}, open(os.path.join(RES57, '_run_info.json'), 'w'))
LOCAL = f'/content/cxrlt_{SIZE_HI}'
READY_X = os.path.join(OUT_DIR, 'cxrlt_518_ready')   # kho do 41c C3-bis tao

FT_BLOCKS = (8, 9, 10, 11)      # mo bang 4 block tren. `11`: I_location ~ 0, nen chon theo chi phi.
LR_FT_DIV = 10.0                # LR cho block mo bang = LR/10 — chuan LP-FT, tranh quen pretrain
BB_RUN = 'a2_E3'; SEEDS = (42,) if CHE_DO_THU else (42, 43)
SEED, SIZE = 42, SIZE_HI
EPOCHS = 6 if CHE_DO_THU else 10
assert SIZE % 14 == 0, f'SIZE {SIZE} KHONG chia het cho 14 (ViT-B/14) -> hong luoi patch'
print(f'DO PHAN GIAI {SIZE} -> {SIZE//14}x{SIZE//14} = {(SIZE//14)**2:,} token | '
      f'{14*350.0/SIZE:.2f} mm/patch   (moc 518: 1,369 token, 9.46 mm)')
print(f'che do THU = {CHE_DO_THU}  ->  bac A {E_BB} epoch | bac B E={EPOCHS} | '
      f'{"1 seed" if CHE_DO_THU else "2 seed"}')
BATCH, ACCUM = 16, 16     # EFF 256 giu nguyen; ha BATCH vi 5,329 token                      # EFF 256 — GIU y het 53/56 de so truc tiep
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
NW = min(16, (os.cpu_count() or 2))
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0
DO_TEST = True      # luu pte_ MOI epoch (bo luat last-3): can de do test o do sau THAT

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

# `BB_SRC` = backbone 518 cua 38a, CHI dung lam KHOI TAO cho bac A.
BB_SRC = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
BB_HI  = os.path.join(RES57, f'backbone_ft_{SIZE_HI}.pt')   # dau ra cua bac A, dung o bac B
P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
for p, hd in ((BB_SRC, '38a'), (P_MICRO, '29c'), (P_VOCAB, '29c'), (P_OOF, '52a')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
_rs = np.random.default_rng(SEED); _su = np.unique(SUB[TR])
_sel = _rs.choice(_su, size=int(round(0.15*len(_su))), replace=False)
SL = TR & np.isin(SUB, _sel)          # van tinh, CHI de bao cao co bao nhieu anh duoc thu hoi
TF_ = TR.copy()                        # <<< R2a: GOP lat chon vao train
print(f'R2a: thu hoi {int(SL.sum()):,} anh lat chon vao train '
      f'({int((TR & ~SL).sum()):,} -> {int(TF_.sum()):,}, +{SL.sum()/max((TR&~SL).sum(),1):.1%})')
print('     LAT CHON GIO NAM TRONG TRAIN -> khong duoc chon bat cu thu gi o luot nay.')

# --- TIER theo dinh nghia CHINH THUC cua giai (paper §3.3: prev tren TRAIN) ---
TIER_CU = TIER
_prev = YM[TR].mean(0)
TIER = {'head':   np.where(_prev > 0.10)[0],
        'medium': np.where((_prev >= 0.01) & (_prev <= 0.10))[0],
        'tail':   np.where(_prev < 0.01)[0]}
print('TIER = CHINH THUC ' + str({k: len(v) for k, v in TIER.items()})
      + ' | TIER_CU (nb28) ' + str({k: len(v) for k, v in TIER_CU.items()}))

# 7 nhan medium gong 71% khoang cach den vo dich (do o nb67) — CONG cua notebook nay
BAY_MEDIUM = ['Infiltration', 'Tortuous Aorta', 'Pleural Thickening', 'Adenopathy',
              'Calcification of the Aorta', 'Fissure', 'Granuloma']
C_7M = np.array([LAB40.index(l) for l in BAY_MEDIUM if l in LAB40])
print(f'7 nhan medium theo doi: {len(C_7M)} nhan | AP TB tren test @518 = 0.125')
print('  (khi doc cong thi LOAI Infiltration: prec 0.261, nhieu nhan, do phan giai khong cuu)')
# R2a: SL GIO NAM TRONG TF_ nen giao cua chung DUNG BANG SL — kiem THUAN chieu do,
# thay vi kiem roi nhau nhu cac luot truoc.
assert int((TF_ & SL).sum()) == int(SL.sum()), 'lat chon CHUA duoc gop het vao train'
assert int(TF_.sum()) == int(TR.sum()), 'TF_ phai bang DUNG TR o luot R2a'
assert not (set(SUB[TR]) & set(SUB[VA])), 'benh nhan train ro ri sang VAL'
assert not (set(SUB[TR]) & set(SUB[TE])), 'benh nhan train ro ri sang TEST'
I_TF, I_SL, I_VA, I_TE = (np.where(m)[0] for m in (TF_, SL, VA, TE))
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,} | '
      f'test {len(I_TE):,}')
print(f'mo bang block {FT_BLOCKS} (LR/{LR_FT_DIV:.0f}) | E={EPOCHS} | '
      f'BATCH {BATCH}x{ACCUM}={BATCH*ACCUM} | seeds {SEEDS}')

Mounted at /content/drive
DO PHAN GIAI 1022 -> 73x73 = 5,329 token | 4.79 mm/patch   (moc 518: 1,369 token, 9.46 mm)
che do THU = True  ->  bac A 3 epoch | bac B E=6 | 1 seed
thiet bi: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM 102 GB | AMP torch.bfloat16
R2a: thu hoi 39,086 anh lat chon vao train (219,785 -> 258,871, +17.8%)
     LAT CHON GIO NAM TRONG TRAIN -> khong duoc chon bat cu thu gi o luot nay.
TIER = CHINH THUC {'head': 9, 'medium': 15, 'tail': 16} | TIER_CU (nb28) {'head': 13, 'medium': 16, 'tail': 11}
7 nhan medium theo doi: 7 nhan | AP TB tren test @518 = 0.125
  (khi doc cong thi LOAI Infiltration: prec 0.261, nhieu nhan, do phan giai khong cuu)
377,110 anh | khop 258,871 | chon 39,086 | val 39,293 | test 78,946
mo bang block (8, 9, 10, 11) (LR/10) | E=6 | BATCH 16x16=256 | seeds (42,)


## C1 · EHR + túi vi khuẩn + XGB + phim trước — verbatim `53`/`56`

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
_m = np.nan_to_num((E_DF.set_index('study_id')[COLS].reindex(_sid)
                    .apply(pd.to_numeric, errors='coerce').to_numpy('float32')), nan=0.0)
# Cot NHI PHAN thi KHONG z-score: p = 1/219,785 -> gia tri chuan hoa 468 -> softmax bao hoa -> NaN
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TF_].mean(0, keepdims=True); _sd = _m[TF_].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_keep = [i for i, c in enumerate(COLS) if not re.match(r'^micro_org_', c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
COLS = [COLS[i] for i in _keep]; EHR_COL = E_ALL[:, _keep]

_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
_VOC = json.load(open(P_VOCAB)); V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b

XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C) and np.isfinite(XGF).all()
_l = np.log(np.clip(XGF, 1e-6, 1-1e-6) / (1 - np.clip(XGF, 1e-6, 1-1e-6)))
XGB_F = np.clip((_l - _l[TF_].mean(0, keepdims=True)) / (_l[TF_].std(0, keepdims=True) + 1e-6),
                -EHR_CLIP, EHR_CLIP).astype('float32')
FEAT = np.concatenate([EHR_COL, XGB_F], 1)      # `both` = 217 cot EHR + 40 cot XGB (53: tot nhat)
print(f'FEAT {FEAT.shape} = {EHR_COL.shape[1]} EHR + {C} XGB | tui vi khuan V={V_MIC-1} dai {MIC_L}')

_BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
_BASE['index_time'] = pd.to_datetime(_BASE.index_time)
_B = (_BASE[_BASE.study_id.isin(set(ORDER.study_id.unique()))]
      .sort_values(['subject_id','index_time']).reset_index(drop=True))
_B['prev_study'] = _B.groupby('subject_id', sort=False).study_id.shift(1)
_PV = dict(zip(_B.study_id.to_numpy(), _B.prev_study.to_numpy()))
_rw = {}
for _i, _s in enumerate(_sid): _rw.setdefault(_s, []).append(_i)
PREV_ROW = np.full(len(ORDER), -1, 'int64')
for _i, _s in enumerate(_sid):
    _p2 = _PV.get(_s, np.nan)
    if not (isinstance(_p2, float) and np.isnan(_p2)): PREV_ROW[_i] = _rw[int(_p2)][0]
_hp = PREV_ROW >= 0
print(f'anh co phim truoc: {_hp.sum():,} / {len(ORDER):,} ({_hp.mean():.1%})  (44a do 67.2%)')
assert _hp.mean() > 0.5 and (PREV_ROW[_hp] != np.arange(len(ORDER))[_hp]).all()

FEAT (377110, 257) = 217 EHR + 40 XGB | tui vi khuan V=305 dai 10
anh co phim truoc: 253,283 / 377,110 (67.2%)  (44a do 67.2%)


## C2 · Ảnh + backbone + **cổng cắt đôi forward**

`Dinov2Layer` trả về **Tensor trần** ở bản `transformers` này, không phải tuple. `layer(h)[0]` sẽ
lấy **mẫu đầu của batch** và broadcasting che lấp lỗi nên không báo gì — `39` đã dính đúng chỗ này.
`_out()` kiểm `isinstance` trước khi lấy `[0]`.

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T
import shutil

os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
# Uu tien KHO da resize san (41c C3-bis tao): copy + bung, vai phut. Neu chua co thi stage tu shard
# goc 1022px — cham hon ~10 lan vi phai giai ma + resize 377k anh.
# Kho dung san CHI co ban 518. O do phan giai khac phai stage tu shard GOC 1022.
if SIZE == 518 and _n < len(ORDER) * 0.999 and os.path.isdir(READY_X):
    for f in sorted(x for x in os.listdir(READY_X) if x.endswith('.tar')):
        src = os.path.join(READY_X, f); tmp = os.path.join('/content', f)
        try:
            shutil.copy(src, tmp)                        # copy TRUOC — khong mo tar tren mount
            assert os.path.getsize(tmp) == os.path.getsize(src), 'copy NGAN tu Drive'
            with tarfile.open(tmp) as tar: tar.extractall(LOCAL, filter='data')
        except Exception as e:
            print(f'  {f} HONG ({type(e).__name__}) -> bo qua')
        if os.path.exists(tmp): os.remove(tmp)
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
    print(f'hydrate tu kho: {_n:,} / {len(ORDER):,}')
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
    assert os.path.isdir(SHARD_DIR), f'khong thay CA {READY_X} lan {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        try:
            if SIZE == SIZE_SHARD:     # shard DA dung kich thuoc -> ghi THANG, khoi
                tmp = dst + '.tmp'     # giai ma + ma hoa lai 377k anh (tiet kiem PHAN LON thoi gian)
                with open(tmp, 'wb') as _f: _f.write(blob)
                os.replace(tmp, dst); return
            im = Image.open(_io.BytesIO(blob)).convert('L')
            if im.size != (SIZE, SIZE):
                im = im.resize((SIZE, SIZE), Image.BICUBIC)
        except Exception:
            return                                       # mot anh hong KHONG duoc giet ca lan chay
        tmp = dst + '.tmp'; im.save(tmp, 'JPEG', quality=95); os.replace(tmp, dst)   # NGUYEN TU
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()     # doc BULK; lap tren mount FUSE se TREO
            assert len(buf) == os.path.getsize(tp), f'{os.path.basename(tp)}: doc NGAN tren mount'
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh'
print(f'anh local: {_n:,} / {len(ORDER):,}')

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

_ck = torch.load(BB_SRC, map_location='cpu', weights_only=False)
_sd0 = {k: v.clone() for k, v in AutoModel.from_pretrained('microsoft/rad-dino').state_dict().items()}
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
backbone.load_state_dict(_ck['bb'], strict=True)
with torch.no_grad():
    _d = max((_sd0[k] - v).abs().max().item() for k, v in backbone.state_dict().items())
del _sd0
assert _d > 1e-6, f'backbone GIONG HET pretrain (lech {_d:.1e}) -> `ft` khong duoc nap'
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
LAYERS = backbone.encoder.layer; NL = len(LAYERS)
GRAD_FROM = min(FT_BLOCKS)
assert max(FT_BLOCKS) < NL and list(FT_BLOCKS) == list(range(GRAD_FROM, NL)), (
    f'FT_BLOCKS {FT_BLOCKS} phai la day LIEN TUC den block cuoi ({NL-1}) — neu khong thi phan '
    'dong cung o TREN chung khong dung chung duoc voi arm ft.')
_cfg = backbone.config
_dp = (getattr(_cfg, 'attention_probs_dropout_prob', 0), getattr(_cfg, 'hidden_dropout_prob', 0),
       getattr(_cfg, 'drop_path_rate', 0))
print(f'BACKBONE `ft` 38a/{BB_RUN} | {NL} block | d={D_CXR} | lech pretrain {_d:.3f}')
print(f'dropout (attn, hidden, drop_path) = {_dp}'
      + ('  -> bang 0, che do train/eval khong anh huong' if max(_dp) == 0 else
         '  -> KHAC 0: ban sao PHAI chay .eval() de khop ban goc'))

def _out(r):
    '''Tensor TRAN, khong phai tuple: `r[0]` se lay MAU DAU cua batch va broadcasting che lap loi.'''
    return r[0] if isinstance(r, tuple) else r

class DS(Dataset):
    '''Tra ve CA HAI anh: hien tai va PHIM TRUOC (+ `has`). `ehr_perm` cho TRAO NHAN QUA.'''
    def __init__(s, idx, tf, ehr_perm=None): s.idx, s.tf, s.perm = np.asarray(idx), tf, ehr_perm
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return s.tf(Image.merge('RGB', (im, im, im)))
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i]); x = s._img(i)
        return dict(image=x, image_prev=(s._img(pr) if pr >= 0 else torch.zeros_like(x)),
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), pos=k,
                    y=torch.from_numpy(YM[i]), row=i,
                    erow=(i if s.perm is None else int(s.perm[k])))

def loader(idx, tf, shuffle, bs, perm=None, nw=NW, drop_last=None):
    return DataLoader(DS(idx, tf, perm), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle if drop_last is None else drop_last,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

def run_blocks(h, lo, hi, layers=None):
    L = LAYERS if layers is None else layers
    for i in range(lo, hi): h = _out(L[i - (0 if layers is None else lo)](h))
    return h

# --- cong 1: forward CAT DOI phai bang forward NGUYEN VEN ---
_b0 = next(iter(loader(I_VA[:4], eval_tf, False, 4, nw=0))); _x = _b0['image'].to(dev)
with torch.no_grad():
    _full = backbone(pixel_values=_x).last_hidden_state
    _split = backbone.layernorm(run_blocks(run_blocks(backbone.embeddings(_x), 0, GRAD_FROM),
                                           GRAD_FROM, NL))
_e1 = float((_full - _split).abs().max())
assert _e1 < 1e-3, (f'forward CAT DOI lech {_e1:.2e} -> da cat SAI cho trong kien truc HF. '
                    'DUNG chay tiep, moi so sau day vo nghia.')
N_TOK = _full.shape[1] - 1
assert N_TOK == (SIZE // backbone.config.patch_size) ** 2
print(f'cong 1: forward cat doi khop toi {_e1:.1e} | {N_TOK} patch token')

stage:   0%|          | 0/378 [00:00<?, ?tar/s]

anh local: 377,110 / 377,110


preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

BACKBONE `ft` 38a/a2_E3 | 12 block | d=768 | lech pretrain 0.021
dropout (attn, hidden, drop_path) = (0.0, 0.0, 0.0)  -> bang 0, che do train/eval khong anh huong
cong 1: forward cat doi khop toi 0.0e+00 | 5329 patch token


## C3 · Kiến trúc — verbatim `53`/`56`

In [ ]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class MicroBag(nn.Module):
    '''`const=True`: MOT chi so, khong phai MIC_L ban sao (tui that TB 1.7 vi khuan).'''
    def __init__(s, V, d, const=False):
        super().__init__()
        s.const = const; s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w):
        if s.const:
            ids = torch.zeros(ids.shape[0], 1, dtype=torch.long, device=ids.device)
            w = torch.ones(ids.shape[0], 1, device=ids.device)
        return s.mlp(F.embedding_bag(ids, s.w, per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    '''d_in token COT + 1 TONG HOP + 1 BAG. Chuoi nay CHUA k1 nhu truong hop rieng.'''
    def __init__(s, d_in, d=256, const=False, V=None):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.const = const
        if const: s.c = nn.Parameter(torch.randn(d_in + 1, d) * 0.02)
        s.mb = MicroBag(V, d, const) if V else None
        if s.mb is not None:
            s.bln = nn.LayerNorm(d); s.cln = nn.LayerNorm(d) if const else nn.Identity()
    def forward(s, e, mids=None, mcnt=None):
        if s.const: base = s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        else:
            col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
            base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([s.cln(base), s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class Tier1(nn.Module):
    '''Q = 1369 PATCH, K/V = token EHR. `pr` zero-init; `ln_a` chuan hoa DAU VAO cua phep chieu
       (38d: thieu no thi arm doi chung sup 0.2416 -> 0.0681).'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))

class Tier1Img(nn.Module):
    '''Phim truoc o muc PATCH, chay o khong gian D_CXR (TRUOC `pin`).
       KHONG tru thang `patch - prior`: hai phim khong duoc dang ky (tu the, do hit vao). '''
    def __init__(s, d, nhead=8, drop=0.1, diff=True):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.nq = nn.LayerNorm(d); s.nk = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d)
        # LAYERSCALE. `pr` zero-init cho dong nhat thuc o buoc 0 nhung KHONG chan duoc toc do
        # lon len: AdamW cap nhat ~lr/buoc/tham so BAT KE gradient nho the nao, nen mot nhanh xuat
        # phat tu 0 lon TUYEN TINH theo buoc x LR. Do `ln_a` chuan hoa `patch - a` (thuong gan 0,
        # phan lon la nhieu) ve chuan don vi, bien do dau ra do TRON VEN `‖pr‖` quyet dinh. Khi no
        # at `patch` thi ML-Decoder doi che do dot ngot -> sup. Da xay ra 3 lan, luon o ep2-4 tuc
        # ngay sau dinh warmup (0.05 x E = nua epoch 1), va `clip` do 0.0% vi gradient VAN binh
        # thuong — van de o BUOC DI, khong o gradient.
        # Nay `pr` khoi tao CHUAN va mot he so LayerScale `1e-4` giu phan du gan 0 luc dau; ca hai
        # deu co gradient that, va he so phai HOC de lon len thay vi lon lên vi quan tinh AdamW.
        s.pr = nn.Linear(d, d)
        s.gamma = nn.Parameter(torch.full((d,), 1e-4))
        s.diff = diff
    def forward(s, patch, prior, has):
        a, _ = s.at(s.nq(patch), s.nk(prior), s.nk(prior), need_weights=False)
        z = s.gamma * s.pr(s.ln_a(patch - a if s.diff else a))
        return patch + z * has.view(-1, 1, 1)      # khong co phim truoc -> mask nhanh di han

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = patch, co the da bi Tier1 dieu bien.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, d_ehr_in=0, t1=False):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None
        if t1: s.ein = nn.Linear(d_ehr_in, d)
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None, need_w=False):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None: kv = s.t1(kv, s.ein(etok))
        h, w = s.img(q, kv, need_w); return s.out(h), w

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44*/45/52*/53/56. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

kien truc san sang


## C4 · Ba arm + bản sao block 8–11 + cổng khoá trước

In [ ]:
# Bo `tpb_ft`: 57 da dong huong mo bang (G_ft -0.007..-0.010). Giu `mldec` lam doi chung
# anh-thuan — no dung chung forward backbone nen gan nhu mien phi.
ARMS = {
  'mldec':  dict(ehr=False, img=False, ft=False),
  'tpb':    dict(ehr=True,  img=True,  ft=False),
}
FT_ARMS = [t for t, c in ARMS.items() if c['ft']]

def build_arms(seed=SEED):
    M, E, I, U = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        torch.manual_seed(seed)     # gieo lai TRUOC TUNG arm -> khoi tao giong het nhau
        if cfg['ehr']: E[tag] = ColTokBag(FEAT.shape[1], D_ETOK, False, V=V_MIC)
        M[tag] = MLDecoder(C, D_CXR, D_MLD, d_ehr_in=D_ETOK, t1=cfg['ehr'])
        if cfg['img']: I[tag] = Tier1Img(D_CXR)
        if cfg['ft']:
            # BAN SAO RIENG cua block 8-11 + layernorm. Dung chung thi viec cap nhat chung se
            # doi dac trung cua `mldec` va `tpb` -> pha luon hai doi chung.
            U[tag] = nn.ModuleList([copy.deepcopy(LAYERS[i]) for i in FT_BLOCKS]
                                   + [copy.deepcopy(backbone.layernorm)])
            for p in U[tag].parameters(): p.requires_grad_(True)
    return M.to(dev), E.to(dev), I.to(dev), U.to(dev)

MOD, ETK, IMG, UPF = build_arms()
NP_ = {t: sum(p.numel() for m in ([MOD[t]] + [x[t] for x in (ETK, IMG, UPF) if t in x])
              for p in m.parameters() if p.requires_grad) for t in ARMS}
print(f'{"arm":<12} {"tham so":>12} {"trong do block 8-11":>21}')
for t in ARMS:
    nu = sum(p.numel() for p in UPF[t].parameters()) if t in UPF else 0
    print(f'  {t:<10} {NP_[t]:>12,} {nu:>21,}')

_e = torch.randn(4, FEAT.shape[1], device=dev)
_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev); _mc = torch.zeros(4, MIC_L, device=dev)
_p = torch.randn(4, N_TOK, D_CXR, device=dev); _pp = torch.randn(4, N_TOK, D_CXR, device=dev)
_h1 = torch.ones(4, device=dev); _h0 = torch.zeros(4, device=dev)
tok = lambda t: ETK[t](_e, _mi, _mc)

assert not any(p.requires_grad for p in backbone.parameters()), 'backbone GOC phai dong cung'
assert not MOD['mldec'].q.weight.requires_grad, 'query_embed dang hoc'
with torch.no_grad():
    m = MOD['tpb'].eval(); l1, _ = m(_p, tok('tpb'))
    _k = m.t1; m.t1 = None; l2, _ = m(_p, tok('tpb')); m.t1 = _k
    assert (l1 - l2).abs().max() < 1e-5, 'Tier1 KHONG zero-init'
    for t in IMG:
        # Cong TUONG DOI thay vi tuyet doi: LayerScale cho dong nhat thuc XAP XI (~1e-4), khong
        # bang chinh xac. Va ti le bien do moi la thu ta thuc su quan tam — chinh no vuot 1 la sup.
        _rel = float((IMG[t](_p, _pp, _h1) - _p).norm() / (_p.norm() + 1e-9))
        assert _rel < 1e-3, f'{t}: Tier1Img nhieu {_rel:.2e} luc khoi tao, phai < 1e-3'
print('cong 2: Tier1 va Tier1Img deu zero-init OK')

# --- cong 3 (ban sao block 8-11) DA GO: luot nay khong co arm `ft` nao ---
# Neu ai do them lai arm ft ma quen cong nay thi phai bao ngay, khong duoc im lang.
assert not FT_ARMS, (f'co arm ft {FT_ARMS} nhung cong 3 (kiem deepcopy) da bi go khoi 64. '
                     'Lay lai tu 57 truoc khi chay.')
print('cong 3: da go (khong co arm ft o luot nay)')
with torch.no_grad():
    _ti = IMG['tpb']; _ti.pr.weight.normal_(0, 0.02)
    _r = float((_ti(_p, _pp*100.0, _h1) - _p).abs().max() / ((_ti(_p, _pp, _h1) - _p).abs().max()+1e-9))
    assert (_ti(_p, _pp, _h0) - _p).abs().max() < 1e-6, 'has=0 KHONG cat duoc nhanh phim truoc'
    nn.init.zeros_(_ti.pr.weight)
assert _r < 3.0, f'phong dai K/V 100x -> bien do ra tang {_r:.1f}x, `ln_a` khong chan duoc'
print(f'cong 4: `ln_a` chan bien do OK ({_r:.2f}x) | has=0 cat sach nhanh OK')
MOD, ETK, IMG, UPF = build_arms()

arm               tham so   trong do block 8-11
  mldec         2,517,032                     0
  tpb           7,330,856                     0
cong 2: Tier1 va Tier1Img deu zero-init OK
cong 3: da go (khong co arm ft o luot nay)
cong 4: `ln_a` chan bien do OK (1.00x) | has=0 cat sach nhanh OK


## C5 · Chấm điểm

In [ ]:
from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}
_rs6 = np.random.default_rng(6); _yy = (_rs6.random((3000, C)) < 0.2).astype('float32')
_ss6 = _rs6.random((3000, C)).astype('float32')
_dd6 = float(np.nanmax(np.abs(ap_cols(_yy, _ss6) -
        np.array([average_precision_score(_yy[:, j], _ss6[:, j]) for j in range(C)]))))
assert _dd6 < 1e-6, f'ap_cols lech sklearn {_dd6:.2e}'
print(f'cong 5: ap_cols khop sklearn toi {_dd6:.2e}')

cong 5: ap_cols khop sklearn toi 1.69e-08


## Bậc A · Fine-tune **backbone** ở đúng độ phân giải cao

`38a` fine-tune RAD-DINO ở `518` rồi đông cứng. Ở đây ta khởi tạo **từ chính checkpoint đó** rồi
thích nghi tiếp ở `SIZE_HI`, dùng đúng công thức LP-FT của `38a` (`đầu đọc 5e-4`, `backbone ×0.1`).

Khởi tạo từ `38a` thay vì từ RAD-DINO gốc là để **tiết kiệm**: backbone đã biết nhãn CXR-LT, chỉ
cần học lại thang không gian mới. `3 epoch` đủ cho việc thích nghi; `6 epoch` nếu chạy đầy đủ.

Đầu đọc ở bậc này là **tạm** — huấn luyện xong thì vứt, chỉ giữ backbone. Bậc B mới dựng
`tpb`/`mldec` thật trên backbone đã đông cứng.

Có **resume**: đã tồn tại `backbone_ft_{SIZE}.pt` thì nạp và bỏ qua huấn luyện.

In [ ]:
# ==== BAC A: fine-tune BACKBONE o SIZE cao (khoi tao tu 38a/a2_E3) ====
import shutil
_t0_bb = time.time()

if os.path.exists(BB_HI):
    _w = torch.load(BB_HI, map_location='cpu', weights_only=False)
    backbone.load_state_dict(_w['bb'], strict=True)
    backbone = backbone.to(dev).eval()
    for p in backbone.parameters(): p.requires_grad_(False)
    print(f'BAC A: nap san {BB_HI}')
    print(f'  da fine-tune {_w.get("ep_done", "?")} epoch o SIZE {_w.get("size", "?")}'
          f' | val macro {_w.get("val_macro", float("nan")):.4f}')
else:
    print(f'BAC A: fine-tune backbone o {SIZE} ({(SIZE//14)**2:,} token), {E_BB} epoch')
    print(f'  BATCH_BB {BATCH_BB} x ACCUM_BB {ACCUM_BB} = EFF {BATCH_BB*ACCUM_BB} (38a dung 240)')

    class DS_BB(Dataset):
        """CHI anh hien tai — bac A khong can phim truoc, tiet kiem mot nua chi phi doc anh."""
        def __init__(s, idx, tf): s.idx, s.tf = np.asarray(idx), tf
        def __len__(s): return len(s.idx)
        def __getitem__(s, k):
            i = int(s.idx[k])
            im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
            return dict(image=s.tf(Image.merge('RGB', (im, im, im))),
                        y=torch.from_numpy(YM[i]), pos=k)

    def _ld_bb(idx, tf, sh, bs, nw=NW):
        return DataLoader(DS_BB(idx, tf), batch_size=bs, shuffle=sh, num_workers=nw,
                          pin_memory=True, drop_last=sh,
                          persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

    for p in backbone.parameters(): p.requires_grad_(True)
    torch.manual_seed(SEED)
    HEAD_BB = MLDecoder(C, D_CXR, D_MLD).to(dev)      # dau doc TAM, huan luyen xong thi vut
    crit_bb = ASL()
    _bp = [p for p in backbone.parameters() if p.requires_grad]
    opt_bb = torch.optim.AdamW(
        [{'params': [p for p in HEAD_BB.parameters() if p.requires_grad], 'lr': LR_BB_HEAD},
         {'params': _bp, 'lr': LR_BB_HEAD * LR_BB_MULT}], weight_decay=WD)
    _st = math.ceil(len(I_TF) / (BATCH_BB * ACCUM_BB)) * E_BB; _wm = int(WARMUP * _st)
    sch_bb = torch.optim.lr_scheduler.LambdaLR(
        opt_bb, lambda s: s / max(_wm, 1) if s < _wm else
        0.5 * (1 + math.cos(math.pi * (s - _wm) / max(_st - _wm, 1))))
    print(f'  tham so backbone {sum(p.numel() for p in _bp):,} | dau doc tam '
          f'{sum(p.numel() for p in HEAD_BB.parameters()):,}')

    _mv = float('nan')      # resume voi ep0_bb == E_BB thi vong lap khong chay
    P_RES_BB = os.path.join(CKPT, 'bacA_resume.pt'); ep0_bb = 0
    if os.path.exists(P_RES_BB):
        _r = torch.load(P_RES_BB, map_location=dev, weights_only=False)
        assert _r['size'] == SIZE, f'resume o SIZE {_r["size"]} != {SIZE} -> doi RUN_TAG'
        backbone.load_state_dict(_r['bb']); HEAD_BB.load_state_dict(_r['head'])
        opt_bb.load_state_dict(_r['opt']); sch_bb.load_state_dict(_r['sch'])
        ep0_bb = _r['ep'] + 1
        print(f'  resume tu epoch {ep0_bb+1}/{E_BB}')

    DL_BB = _ld_bb(I_TF, train_tf, True, BATCH_BB)
    for ep in range(ep0_bb, E_BB):
        backbone.train(); HEAD_BB.train()
        _t, _tot, _nb = time.time(), 0.0, 0
        opt_bb.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(DL_BB, desc=f'bacA ep{ep+1}/{E_BB}', leave=False)):
            with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
                h = backbone(pixel_values=b['image'].to(dev, non_blocking=True)
                             ).last_hidden_state[:, 1:]
            lg, _ = HEAD_BB(h.float())
            if not torch.isfinite(lg).all():
                opt_bb.zero_grad(set_to_none=True); continue
            loss = crit_bb(lg.float(), b['y'].to(dev))
            (loss / ACCUM_BB).backward(); _tot += float(loss.detach()); _nb += 1
            if (j + 1) % ACCUM_BB == 0:
                gn = torch.nn.utils.clip_grad_norm_(
                    _bp + list(HEAD_BB.parameters()), CLIP, error_if_nonfinite=False)
                if not torch.isfinite(gn): opt_bb.zero_grad(set_to_none=True); continue
                opt_bb.step(); sch_bb.step(); opt_bb.zero_grad(set_to_none=True)

        backbone.eval(); HEAD_BB.eval()
        _PV = np.zeros((len(I_VA), C), 'float32')
        with torch.no_grad():
            for b in tqdm(_ld_bb(I_VA, eval_tf, False, BATCH_BB, nw=0), desc='eval', leave=False):
                with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
                    h = backbone(pixel_values=b['image'].to(dev)).last_hidden_state[:, 1:]
                lg, _ = HEAD_BB(h.float())
                _PV[b['pos'].numpy()] = torch.sigmoid(lg.float()).cpu().numpy()
        _ap = ap_cols(YM[VA], _PV); _mv = float(np.nanmean(_ap))
        print(f'  bacA ep{ep+1}/{E_BB} | {_nb*BATCH_BB/max(time.time()-_t,1e-9):.0f} img/s | '
              f'{(time.time()-_t)/60:.1f} ph | loss {_tot/max(_nb,1):.4f} | '
              f'val macro {_mv:.4f} | ' + ' '.join(f'{t}:{v:.4f}' for t, v in tiers(_ap).items()))
        print(f'    ^ dau doc TAM (khong EHR, khong phim truoc) — chi de THEO DOI backbone,'
              f' KHONG so voi 0.2663')
        ensure_res()
        torch.save(dict(bb=backbone.state_dict(), head=HEAD_BB.state_dict(),
                        opt=opt_bb.state_dict(), sch=sch_bb.state_dict(),
                        ep=ep, size=SIZE), P_RES_BB)

    # --- luu backbone: LOCAL truoc, roi copy len Drive va SO KICH THUOC ---
    _wd = dict(bb=backbone.state_dict(), size=SIZE, ep_done=E_BB, val_macro=_mv,
               init_tu=BB_RUN, lr_head=LR_BB_HEAD, lr_mult=LR_BB_MULT)
    _loc = f'/content/backbone_ft_{SIZE}.pt'
    torch.save(_wd, _loc); _sz = os.path.getsize(_loc)
    try:
        ensure_res(); shutil.copyfile(_loc, BB_HI)
        assert os.path.getsize(BB_HI) == _sz, 'Drive nhan THIEU byte -> ghi CUT'
        print(f'  backbone {_sz/1e6:.0f} MB -> Drive OK: {BB_HI}')
    except Exception as e:
        print(f'  !! KHONG dua duoc backbone len Drive: {type(e).__name__} {e}')
        print(f'     ban LOCAL o {_loc} — copy tay TRUOC khi tat session.')

    backbone = backbone.eval()
    for p in backbone.parameters(): p.requires_grad_(False)
    del HEAD_BB, opt_bb, sch_bb, _bp
    torch.cuda.empty_cache()

# --- cong: luoi patch dung SIZE moi, va backbone da THAT SU doi ---
_b0 = next(iter(loader(I_VA[:4], eval_tf, False, 4, nw=0)))
# `shared_lo`/`upper_frozen` dinh nghia o cell HUAN LUYEN (sau cell nay) -> goi thang backbone.
with torch.no_grad(), torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
    _h = backbone(pixel_values=_b0['image'].to(dev)).last_hidden_state[:, 1:]
assert _h.shape[1] == (SIZE // 14) ** 2, \
    f'luoi patch {_h.shape[1]} != {(SIZE//14)**2} -> SIZE khong chia het cho 14?'
N_TOK = _h.shape[1]
_ck0 = torch.load(BB_SRC, map_location='cpu', weights_only=False)['bb']
with torch.no_grad():
    _d = max((_ck0[k].float() - v.detach().cpu().float()).abs().max().item()
             for k, v in backbone.state_dict().items() if k in _ck0)
del _ck0
assert not any(p.requires_grad for p in backbone.parameters()), 'backbone chua dong cung lai'
print(f'\ncong BAC A: {N_TOK:,} patch token @ {SIZE}px = {14*350.0/SIZE:.2f} mm/patch')
print(f'  lech so voi backbone 518 cua 38a: {_d:.4f}  (0 = bac A khong chay)')
print(f'  backbone DA dong cung lai | bac A ton {(time.time()-_t0_bb)/3600:.2f} h')


BAC A: nap san /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb68/r1022/backbone_ft_1022.pt
  da fine-tune 3 epoch o SIZE 1022 | val macro 0.2517

cong BAC A: 5,329 patch token @ 1022px = 4.79 mm/patch
  lech so voi backbone 518 cua 38a: 0.0664  (0 = bac A khong chay)
  backbone DA dong cung lai | bac A ton 0.00 h


## C6 · Huấn luyện

**Sửa hai lỗi hiệu năng của `56`.** Phần đông cứng (block 8–11 gốc) tính **một lần** cho cả batch
rồi ba arm dùng chung; mọi thứ chạy dưới `autocast`. `56` gọi `upper()` riêng cho từng arm và chạy
fp32 ⇒ 15 img/s. Kỳ vọng ở đây nhanh hơn ~5 lần.

Ghi mỗi epoch: `pr_norm` (‖`Tier1Img.pr`‖ — theo dõi cú sụp), `clip_pct`, và **`drift`** = mức
block 8–11 của arm `ft` đã đi xa bản gốc bao nhiêu.

In [ ]:
crit = ASL(gn=4, gp=0).to(dev)
MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)
FEAT_T = torch.from_numpy(FEAT)
# Lay tham so 2-D dau tien, KHONG phai `next(iter(...))`: tham so dau cua mot `Dinov2Layer` la
# `norm1.weight` (1-D), ma nhom ndim<=1 von da co weight_decay=0 -> do o do thi MU voi chinh thu
# ta muon theo doi. Weight decay chi tac dong len ma tran 2-D.
# Van KHONG bam vao ten thuoc tinh cua HF (`.attention.attention.query`) — gia dinh do da lam hong `39`.
_first2d = lambda m: next(p for p in m.parameters() if p.ndim > 1)
_REF = {i: _first2d(LAYERS[i]).detach().clone() for i in FT_BLOCKS}

@torch.no_grad()
def shared_lo(img):
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return run_blocks(backbone.embeddings(img), 0, GRAD_FROM)

@torch.no_grad()
def upper_frozen(h):
    '''Block 8-11 GOC. Tinh MOT lan cho ca batch — moi arm dong cung dung chung.'''
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return backbone.layernorm(run_blocks(h, GRAD_FROM, NL))[:, 1:].float()

def upper_ft(h, t):
    '''Block 8-11 BAN SAO cua arm `t`, CO graph.'''
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        for k in range(len(FT_BLOCKS)): h = _out(UPF[t][k](h))
        return UPF[t][-1](h)[:, 1:].float()

def arm_forward(t, pf, ppf, hl, hlp, erow, has):
    '''`pf`/`ppf` = dac trung DONG CUNG dung chung. Arm `ft` bo qua chung, tu tinh tu `hl`.'''
    cfg = ARMS[t]
    et = ETK[t](FEAT_T[erow].to(dev), MIDS_T[erow].to(dev), MCNT_T[erow].to(dev)) \
         if cfg['ehr'] else None
    if cfg['ft']: p, pp = upper_ft(hl, t), upper_ft(hlp, t)
    else:         p, pp = pf, ppf
    if cfg['img']: p = IMG[t](p, pp, has)
    return MOD[t](p, et)[0]

@torch.no_grad()
def predict(idx, perm=None, bs=32):
    MOD.eval(); ETK.eval(); IMG.eval(); UPF.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    for b in tqdm(loader(idx, eval_tf, False, bs, perm, nw=0, drop_last=False),
                  desc='eval', leave=False):
        hl = shared_lo(b['image'].to(dev, non_blocking=True))
        hlp = shared_lo(b['image_prev'].to(dev, non_blocking=True)); hs = b['has'].to(dev)
        pf, ppf = upper_frozen(hl), upper_frozen(hlp)
        for t in ARMS:
            P[t][b['pos'].numpy()] = torch.sigmoid(
                arm_forward(t, pf, ppf, hl, hlp, b['erow'], hs).float()).cpu().numpy()
    assert not any(np.isnan(v).any() for v in P.values()), 'predict cho ra NaN'
    return P

P_LOG = os.path.join(RES57, 'epoch_log.json')
LOG = json.load(open(P_LOG)) if os.path.exists(P_LOG) else {}
DL = loader(I_TF, train_tf, True, BATCH)

for seed in SEEDS:
    sk = f's{seed}'
    if LOG.get(sk, {}).get('done'): print(f'{sk}: da xong, bo qua'); continue
    MOD, ETK, IMG, UPF = build_arms(seed)
    PARAM_OF = {t: [p for m in ([MOD[t]] + [x[t] for x in (ETK, IMG, UPF) if t in x])
                    for p in m.parameters() if p.requires_grad] for t in ARMS}
    N_CLIP = {t: 0 for t in ARMS}; N_SKIP = {t: 0 for t in ARMS}; N_STEP = 0
    # Block mo bang dung LR/10 (chuan LP-FT): LR day du se xoa kien thuc pretrain trong vai buoc.
    _ftp = {id(p) for t in UPF for p in UPF[t].parameters()}
    rest = [p for m in (MOD, ETK, IMG) for p in m.parameters() if p.requires_grad]
    ftpp = [p for t in UPF for p in UPF[t].parameters() if p.requires_grad]
    opt = torch.optim.AdamW(
        [{'params': [p for p in rest if p.ndim > 1], 'weight_decay': WD, 'lr': LR},
         {'params': [p for p in rest if p.ndim <= 1], 'weight_decay': 0.0, 'lr': LR},
         # KHONG weight decay tren block pretrain. AdamW tach roi co tham so `w(1-lr*wd)` moi buoc;
         # voi LR/10 thi luc co (~0.4%/4k buoc) manh ngang gradient nhiem vu, nen `drift` do duoc
         # phan lon la BI BAO MON chu khong phai thich nghi — va do dung nhip `G_ft` di xuong.
         {'params': [p for p in ftpp if p.ndim > 1], 'weight_decay': 0.0, 'lr': LR/LR_FT_DIV},
         {'params': [p for p in ftpp if p.ndim <= 1], 'weight_decay': 0.0, 'lr': LR/LR_FT_DIV}])
    steps = math.ceil(len(I_TF) / (BATCH*ACCUM)) * EPOCHS; warm = int(WARMUP*steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    LOG.setdefault(sk, {})
    P_RES = os.path.join(CKPT, f'{sk}_resume.pt'); ep0 = 0
    if os.path.exists(P_RES):
        _r = torch.load(P_RES, map_location=dev, weights_only=False)
        _old = _r.get('arms')
        assert _old is None or list(_old) == list(ARMS), (
            f'checkpoint co {len(_old)} arm {list(_old)}, notebook co {len(ARMS)} arm.\n'
            f'  -> DOI RUN_TAG o C0 (vd "r2") roi chay lai tu C0. KHONG xoa gi.')
        MOD.load_state_dict(_r['mod']); ETK.load_state_dict(_r['etk'])
        IMG.load_state_dict(_r['img']); UPF.load_state_dict(_r['upf'])
        opt.load_state_dict(_r['opt']); sch.load_state_dict(_r['sch'])
        N_CLIP, N_SKIP, N_STEP, ep0 = _r['n_clip'], _r['n_skip'], _r['n_step'], _r['ep'] + 1
        print(f'{sk}: resume tu epoch {ep0+1}/{EPOCHS}')

    for ep in range(ep0, EPOCHS):
        # `UPF` chay EVAL y het ban goc (`backbone` luon o .eval()). No VAN nhan gradient va VAN
        # cap nhat — `eval()` chi tat dropout/drop-path. Neu de `.train()` thi `tpb_ft` co them
        # regularization NGAU NHIEN ma `tpb` khong co, va `G_ft` khong con do rieng "trong so thich
        # nghi" nua. C2 in ra cau hinh dropout de biet rui ro nay co that hay bang 0.
        MOD.train(); ETK.train(); IMG.train(); UPF.eval()
        t0, tot, nb = time.time(), {t: 0.0 for t in ARMS}, 0
        opt.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(DL, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            hl = shared_lo(b['image'].to(dev, non_blocking=True))
            hlp = shared_lo(b['image_prev'].to(dev, non_blocking=True))
            y = b['y'].to(dev); hs = b['has'].to(dev)
            pf, ppf = upper_frozen(hl), upper_frozen(hlp)      # MOT lan cho ca batch
            loss, bad = 0.0, []
            for t in ARMS:
                lg = arm_forward(t, pf, ppf, hl, hlp, b['erow'], hs)
                if not torch.isfinite(lg).all(): bad.append(t); N_SKIP[t] += 1; continue
                l_ = crit(lg.float(), y); loss = loss + l_; tot[t] += float(l_.detach())
            if bad and (sum(N_SKIP.values()) <= 3 or j % 400 == 0):
                print(f'   [{sk} ep{ep+1} buoc {j}] logit khong huu han: {bad}')
            if not torch.is_tensor(loss): opt.zero_grad(set_to_none=True); continue
            (loss / ACCUM).backward(); nb += 1
            if (j + 1) % ACCUM == 0:
                # Kiem gradient TRUOC khi buoc: total_norm = nan cho clip_coef = nan va BOI BAN
                # toan bo tham so -> mot buoc hong lam chet vinh vien.
                for t, ps in PARAM_OF.items():
                    gn = torch.nn.utils.clip_grad_norm_(ps, CLIP, error_if_nonfinite=False)
                    if not torch.isfinite(gn):
                        for p in ps:
                            if p.grad is not None: p.grad.zero_()
                        N_SKIP[t] += 1
                    elif float(gn) > CLIP: N_CLIP[t] += 1
                N_STEP += 1; opt.step(); sch.step(); opt.zero_grad(set_to_none=True)

        PV = predict(I_VA); ensure_res()
        for t, p in PV.items(): np.save(os.path.join(RES57, f'p_{t}_{sk}_ep{ep+1}.npy'), p)
        # KHONG luu `psl_`: lat chon da NAM TRONG train o luot nay, nen du doan tren no la
        # TRONG MAU. Luu ra la moi nguoi (ke ca ta) dung no de chon.
        if DO_TEST:          # MOI epoch — bo luat last-3, de do duoc test o do sau THAT
            PT = predict(I_TE); ensure_res()
            for t, p in PT.items(): np.save(os.path.join(RES57, f'pte_{t}_{sk}_ep{ep+1}.npy'), p)
            with open(P_TOUCH, 'a') as _f:
                _f.write(json.dumps(dict(nb='64', run=RUN_TAG, seed=seed, ep=ep+1, n=len(I_TE),
                                         arms=list(ARMS), muc_dich='luot chot R2a',
                                         t=datetime.datetime.now().isoformat(timespec='seconds')))+'\n')
        with torch.no_grad():
            # TI LE BIEN DO THUC cua nhanh phim truoc: ‖phan du‖ / ‖patch‖. `pr_norm` chi la so
            # GIAN TIEP; day moi la dai luong vuot 1 thi sup. Do tren mot lo val co dinh.
            _bd = next(iter(loader(I_VA[:32], eval_tf, False, 32, nw=0)))
            _hb = shared_lo(_bd['image'].to(dev)); _hbp = shared_lo(_bd['image_prev'].to(dev))
            _hs2 = _bd['has'].to(dev)
            _pb, _ppb = upper_frozen(_hb), upper_frozen(_hbp)
            _ri = {}
            for t in IMG:
                _src = upper_ft(_hb, t) if ARMS[t]['ft'] else _pb
                _srp = upper_ft(_hbp, t) if ARMS[t]['ft'] else _ppb
                _ri[t] = float(((IMG[t](_src, _srp, _hs2) - _src).norm(dim=-1)
                                / (_src.norm(dim=-1) + 1e-9)).mean())
            _dr = {t: float(max((_first2d(UPF[t][k]) - _REF[i]).norm()
                                / (_REF[i].norm() + 1e-9) for k, i in enumerate(FT_BLOCKS)))
                   for t in UPF}
            # `drift` mot minh khong phan biet duoc BI CO LAI voi XOAY HUONG. `wnorm` = ‖w‖/‖w_goc‖:
            # < 1 la bi bao mon, ~1 ma drift van tang la thich nghi that.
            _wn = {t: float(np.mean([float(_first2d(UPF[t][k]).norm() / (_REF[i].norm() + 1e-9))
                                     for k, i in enumerate(FT_BLOCKS)])) for t in UPF}
        _e2 = {}
        for t, p in PV.items():
            ap = ap_cols(YM[VA], p)
            # KHONG co `sel`: lat chon nam trong train o luot nay (xem C0)
            _e2[t] = dict(mAP_anh=float(np.nanmean(ap)), **tiers(ap),
                          pr_norm=float(IMG[t].pr.weight.norm()) if t in IMG else 0.0,
                          clip_pct=N_CLIP[t]/max(N_STEP, 1), drift=_dr.get(t, 0.0),
                          ratio_img=_ri.get(t, 0.0), wnorm=_wn.get(t, 0.0))
        LOG[sk][str(ep)] = _e2; json.dump(LOG, open(P_LOG, 'w'))
        torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), img=IMG.state_dict(),
                        upf=UPF.state_dict(), opt=opt.state_dict(), sch=sch.state_dict(),
                        n_clip=N_CLIP, n_skip=N_SKIP, n_step=N_STEP, ep=ep, seed=seed,
                        arms=list(ARMS)), P_RES)
        # ---- ANH CHUP TRONG SO tung epoch, TEN RIENG nen khong bao gio bi ghi de ----
        # 53/r3 mat sach trong so epoch 2-10: no ghi de MOT file resume 550 MB moi epoch qua
        # Drive FUSE, va cac lan ghi sau epoch 1 that bai IM LANG. Con lai duy nhat ban ep1.
        # File nay KHONG co optimizer (~40 MB), ghi ra dia LOCAL truoc (luon duoc) roi copy
        # len Drive va SO KICH THUOC — ghi hong thi bao ngay thay vi phat hien sau nhieu tuan.
        import shutil          # C2 co import nhung NAM TRONG nhanh staging anh (khong chay khi
                               # anh da cache) -> import lai o day cho chac.
        _wd = dict(mod=MOD.state_dict(), etk=ETK.state_dict(), img=IMG.state_dict(),
                   ep=ep, seed=seed, arms=list(ARMS), run=RUN_TAG)
        _loc = f'/content/w_{sk}_ep{ep+1}.pt'
        torch.save(_wd, _loc); _sz = os.path.getsize(_loc)
        _pw = os.path.join(CKPT, f'{sk}_w_ep{ep+1}.pt')
        try:
            shutil.copyfile(_loc, _pw)
            assert os.path.getsize(_pw) == _sz, (f'Drive nhan {os.path.getsize(_pw):,} B '
                                                 f'thay vi {_sz:,} B -> ghi CUT')
            print(f'   trong so ep{ep+1}: {_sz/1e6:.0f} MB -> Drive OK')
        except Exception as _e9:
            print(f'   !! KHONG dua duoc trong so ep{ep+1} len Drive: {type(_e9).__name__} {_e9}')
            print(f'      Ban LOCAL van con o {_loc} — copy tay TRUOC khi tat session.')
        g = lambda a, b: _e2[a]['mAP_anh'] - _e2[b]['mAP_anh']
        print(f'{sk} ep{ep+1:>2} | {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | '
              f'{(time.time()-t0)/60:.1f} ph | loss ' +
              ' '.join(f'{t}:{tot[t]/max(nb,1):.3f}' for t in ARMS))
        print('   VAL      ' + ' '.join(f'{t:>9}:{_e2[t]["mAP_anh"]:.4f}' for t in ARMS))
        print('   TAIL     ' + ' '.join(f'{t:>9}:{_e2[t]["tail"]:.4f}' for t in ARMS))
        print('   ^ VAL chi de THEO DOI suc khoe — E=10 da KHOA, khong duoc doi gi theo dong nay')
        print(f'   G_ctx = tpb - mldec   {g("tpb","mldec"):+.4f}   (EHR + phim truoc; '
              f'tail {_e2["tpb"]["tail"]-_e2["mldec"]["tail"]:+.4f})')
        print('   ratio_img ' + ' '.join(f'{t}:{_ri[t]:.4f}' for t in IMG)
              + '   (vuot ~1 = phan du AT patch -> sup)')
        print('   pr_norm ' + ' '.join(f'{t}:{_e2[t]["pr_norm"]:.2f}' for t in ARMS if t in IMG)
              + f' | clip {max(v["clip_pct"] for v in _e2.values()):.1%}')
        if max(N_SKIP.values()) > 0:
            print('   BUOC BI BO: ' + ' '.join(f'{t}:{v}' for t, v in N_SKIP.items() if v > 0))
    ensure_res()
    torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), img=IMG.state_dict(),
                    upf=UPF.state_dict(), seed=seed, arms=list(ARMS)),
               os.path.join(CKPT, f'{sk}.pt'))
    LOG[sk]['done'] = True; json.dump(LOG, open(P_LOG, 'w'))
    print(f'{sk}: xong')
print('tat ca seed xong')

s42 ep1/6:   0%|          | 0/16179 [00:00<?, ?it/s]

eval:   0%|          | 0/1228 [00:00<?, ?it/s]

eval:   0%|          | 0/2468 [00:00<?, ?it/s]

/tmp/ipykernel_4141/2000598729.py:163: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  pr_norm=float(IMG[t].pr.weight.norm()) if t in IMG else 0.0,


   trong so ep1: 40 MB -> Drive OK
s42 ep 1 | 20 img/s | 217.7 ph | loss mldec:0.040 tpb:0.039
   VAL          mldec:0.2440       tpb:0.2505
   TAIL         mldec:0.1102       tpb:0.1168
   ^ VAL chi de THEO DOI suc khoe — E=10 da KHOA, khong duoc doi gi theo dong nay
   G_ctx = tpb - mldec   +0.0065   (EHR + phim truoc; tail +0.0066)
   ratio_img tpb:0.1669   (vuot ~1 = phan du AT patch -> sup)
   pr_norm tpb:18.85 | clip 0.0%


...

   trong so ep5: 40 MB -> Drive OK
s42 ep 5 | 20 img/s | 213.6 ph | loss mldec:0.036 tpb:0.035
   VAL          mldec:0.2493       tpb:0.2527
   TAIL         mldec:0.1197       tpb:0.1164
   ^ VAL chi de THEO DOI suc khoe — E=10 da KHOA, khong duoc doi gi theo dong nay
   G_ctx = tpb - mldec   +0.0034   (EHR + phim truoc; tail -0.0033)
   ratio_img tpb:0.2683   (vuot ~1 = phan du AT patch -> sup)
   pr_norm tpb:23.06 | clip 0.0%


s42 ep6/6:   0%|          | 0/16179 [00:00<?, ?it/s]

eval:   0%|          | 0/1228 [00:00<?, ?it/s]

eval:   0%|          | 0/2468 [00:00<?, ?it/s]

   trong so ep6: 40 MB -> Drive OK
s42 ep 6 | 20 img/s | 214.0 ph | loss mldec:0.035 tpb:0.034
   VAL          mldec:0.2496       tpb:0.2524
   TAIL         mldec:0.1205       tpb:0.1170
   ^ VAL chi de THEO DOI suc khoe — E=10 da KHOA, khong duoc doi gi theo dong nay
   G_ctx = tpb - mldec   +0.0029   (EHR + phim truoc; tail -0.0035)
   ratio_img tpb:0.2717   (vuot ~1 = phan du AT patch -> sup)
   pr_norm tpb:23.17 | clip 0.0%
s42: xong
tat ca seed xong


## C7 · Kết quả + cổng

In [2]:
# ==== BAC C (TU DU) — so 68 @1022 voi 64 @518, cung thang tier va cung do sau epoch ====
import os, json, glob, re
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
SIZE_HI, RUN_HI, EPOCHS, SEEDS = 1022, 'r1022', 6, (42,)

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); TR, VA = SPLIT == 'train', SPLIT == 'val'; Y_VA = YM[VA]

def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o

prev = YM[TR].mean(0)
TIER = {'head': np.where(prev > 0.10)[0],
        'medium': np.where((prev >= 0.01) & (prev <= 0.10))[0],
        'tail': np.where(prev < 0.01)[0]}
BAY_MEDIUM = ['Infiltration', 'Tortuous Aorta', 'Pleural Thickening', 'Adenopathy',
              'Calcification of the Aorta', 'Fissure', 'Granuloma']
C_7M = np.array([LAB40.index(l) for l in BAY_MEDIUM if l in LAB40])
print('tier CHINH THUC ' + str({k: len(v) for k, v in TIER.items()}))

def _ens(goc, run, arm, eps_toi, seeds):
    base, nm, uq = os.path.dirname(os.path.join(OUT_DIR, goc)), os.path.basename(goc), {}
    for d in os.listdir(base):
        if not (d == nm or d.startswith(nm + ' (')): continue
        p = os.path.join(base, d, run)
        if not os.path.isdir(p): continue
        for f in sorted(glob.glob(os.path.join(p, f'p_{arm}_s*_ep*.npy'))):
            m = re.search(r'_s(\d+)_ep(\d+)\.npy$', os.path.basename(f))
            if m and int(m.group(2)) <= eps_toi and int(m.group(1)) in seeds:
                uq.setdefault(os.path.basename(f), f)
    if not uq: return None, 0
    P = np.mean([np.load(f).astype('float32') for f in uq.values()], 0)
    assert P.shape == (int(VA.sum()), C), f'{goc}: hinh dang {P.shape}'
    return P, len(uq)

P68, n68 = _ens('stage4_nb68', RUN_HI, 'tpb', EPOCHS, set(SEEDS))
P64, n64 = _ens('stage4_nb64', 'r2a',  'tpb', EPOCHS, set(SEEDS))
print(f'ensemble CUNG do sau (ep<={EPOCHS}, seed {sorted(SEEDS)}): '
      f'68@{SIZE_HI} {n68} file | 64@518 {n64} file')
assert P68 is not None and P64 is not None, 'thieu mot ben'
a68, a64 = ap_cols(Y_VA, P68), ap_cols(Y_VA, P64)

print(f'\n{"":<12}{"64 @518":>10}{"68 @"+str(SIZE_HI):>11}{"chenh":>9}')
for nm_, cols in [('macro', np.arange(C))] + [(t, TIER[t]) for t in ('head', 'medium', 'tail')]:
    x, y = float(np.nanmean(a64[cols])), float(np.nanmean(a68[cols]))
    print(f'  {nm_:<10}{x:>10.4f}{y:>11.4f}{y-x:>+9.4f}')

print(f'\n=== CONG: 7 nhan medium ===')
print(f'{"nhan":<28}{"64 @518":>10}{"68 @"+str(SIZE_HI):>11}{"chenh":>9}{"prev%":>8}')
for j in C_7M[np.argsort(a64[C_7M])]:
    print(f'  {LAB40[j]:<26}{a64[j]:>10.4f}{a68[j]:>11.4f}{a68[j]-a64[j]:>+9.4f}{prev[j]*100:>8.3f}'
          + ('   <- nhan BAN (prec 0.26), loai khoi cong' if LAB40[j] == 'Infiltration' else ''))
_c6 = np.array([j for j in C_7M if LAB40[j] != 'Infiltration'])
d7 = float(np.nanmean(a68[_c6]) - np.nanmean(a64[_c6])); dM = float(np.nanmean(a68) - np.nanmean(a64))
print(f'\n  6 nhan (bo Infiltration): {np.nanmean(a64[_c6]):.4f} -> {np.nanmean(a68[_c6]):.4f}'
      f'   {d7:+.4f}   (cong: >= +0.03)')
print(f'  macro toan bo           : {np.nanmean(a64):.4f} -> {np.nanmean(a68):.4f}   {dM:+.4f}')

print(f'\n=== 10 nhan LOI nhat / HAI nhat khi len {SIZE_HI} ===')
_o = np.argsort(a68 - a64); _t = {j: t for t, c in TIER.items() for j in c}
for ten, idx in (('HAI nhat', _o[:10]), ('LOI nhat', _o[::-1][:10])):
    print(f'  --- {ten} ---')
    for j in idx:
        print(f'    {LAB40[j]:<26}{a64[j]:>9.4f} -> {a68[j]:<9.4f}{a68[j]-a64[j]:>+9.4f}'
              f'  {_t[j]:<7} prev {prev[j]*100:6.3f}%')

print('\n' + '=' * 74)
if d7 >= 0.03 and dM >= -0.0007:
    print(f'CONG DAT. -> chay day du: CHE_DO_THU=False (A 6 epoch, B E=10, 2 seed)')
elif d7 > 0.0007:
    print(f'CO tin hieu ({d7:+.4f}) nhung DUOI cong +0.03. Chi phi 7.27x co dang khong?')
else:
    print(f'CONG TRUOT ({d7:+.4f}). Do phan giai KHONG nang duoc nhom nhan manh.')
    print('  CAVEAT: bac A chi 3 epoch thich nghi (38a dung 6 o 518), 1 seed, E=6.')
print('=' * 74); print('CHUA cham test.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
tier CHINH THUC {'head': 9, 'medium': 15, 'tail': 16}
ensemble CUNG do sau (ep<=6, seed [42]): 68@1022 6 file | 64@518 6 file

               64 @518   68 @1022    chenh
  macro         0.2645     0.2590  -0.0055
  head          0.5593     0.5545  -0.0048
  medium        0.2227     0.2256  +0.0029
  tail          0.1379     0.1241  -0.0138

=== CONG: 7 nhan medium ===
nhan                           64 @518   68 @1022    chenh   prev%
  Infiltration                  0.0656     0.0649  -0.0007   3.897   <- nhan BAN (prec 0.26), loai khoi cong
  Tortuous Aorta                0.0820     0.0778  -0.0042   1.289
  Adenopathy                    0.0902     0.0941  +0.0039   1.317
  Pleural Thickening            0.1040     0.0932  -0.0108   1.264
  Fissure                       0.1166     0.1213  +0.0047   1.083
  Granuloma                     0.1331     0.1815  +0.04